In [ ]:
#!pip install boto3 duckdb
#!pip install duckdb

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:
import boto3

s3 = boto3.client(
    service_name="s3",
    endpoint_url=os.getenv("ENDPOINT_URL"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name="auto",
)

In [4]:
#s3.upload_file(r"C:\Users\revan\Downloads\images.jpg","mybucket","mypic.jpg")

In [5]:
#s3.download_file("mybucket","mypic.jpg",r"C:\Users\revan\Downloads\CSV_Analysis_code\mygod.jpeg")

In [6]:
def upload_file_s3(path,name):
    s3.upload_file(path,"mybucket",name)


In [7]:
import pandas as pd
import json

def getdiscribtions(pathfile,filename):

    try:
        if filename.lower().endswith(".csv"):
            file = pd.read_csv(pathfile)

        elif filename.lower().endswith(".xlsx"):
            file = pd.read_excel(pathfile)

        else:
            return "Only CSV and XLSX files are supported"

    except Exception as e:
        return f"Failed to read file: {e}"


    file = pd.read_csv(pathfile)
    s3_bucket_path = f"s3://mybucket/{filename}"

    schema = {"filename":s3_bucket_path,
                "rows":len(file),
                "columns":[]}


    for col in file.columns:

        dtype = str(file[col].dtype)

        if dtype == object:
            file[col] = file[col].apply(lambda x:x.lower())

            schema['columns'].append(
                    {
                    "name":col,
                    "dtype":str(file[col].dtype),
                })
                               
        else:

                schema['columns'].append(
                {
                    "name":col,
                    "dtype":str(file[col].dtype),
                }
            )
    return schema

In [8]:
from botocore.exceptions import ClientError

def file_exist(filename:str,bucket:str="mybucket"):

    try:
        s3.head_object(Bucket = bucket,Key = filename)
        return True

    except ClientError as e:
        if e.response['Error']['Code'] == "404":
            return False
        else:
            print(f"An error occurred: {e}")
            raise

In [9]:
path = r"C:\Users\revan\Downloads\CSV_Analysis_code\relational_csv_dataset"

file_describtion = []
for i in os.listdir(path):
    
   file_describtion.append(getdiscribtions(os.path.join(path,i),i))
   if not file_exist(i):
      print(f"we are uploading {i}")
      upload_file_s3(os.path.join(path,i),i)

In [23]:
os.listdir("./storage")

['categories.csv',
 'customers.csv',
 'orders.csv',
 'order_items.csv',
 'products.csv']

In [10]:
import duckdb


con = duckdb.connect()

con.execute("INSTALL httpfs; LOAD httpfs;")

In [11]:
access_key = os.getenv("AWS_ACCESS_KEY_ID")
secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
endpoint = os.getenv("ENDPOINT_URL")

endpoint = endpoint.removeprefix("https://").removesuffix("/")

con.execute(f"""
CREATE SECRET r2_secret (
    TYPE S3,
    KEY_ID {access_key},
    SECRET {secret_key},
    REGION 'auto',
    ENDPOINT '{endpoint}',
    URL_STYLE 'path'
);
""")

In [12]:
con.execute("FROM duckdb_secrets()").fetchdf()

,name,type,provider,persistent,storage,scope,secret_string
0,r2_secret,s3,config,False,memory,"[s3://, s3n://, s3a://]",name=r2_secret;type=s3;provider=config;seriali...


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_core.messages import SystemMessage,HumanMessage 
from IPython.display import Markdown

In [14]:
import seaborn as sns
import matplotlib.pyplot as plt
@tool
def get_details_from_query(query):
    """this is a function that which will execute the query that which 
    will give all the answers in text format not the plots"""


    try:
        result = con.execute(f"""{query}""").fetchdf()
        explaination = llm.invoke(f"""
        SQL:{query}
        Query result:{result.to_string(index=False)}
        
        Explain the result shown in the chart.
        Mention important trends, highest/lowest values,
        percentages if relevant, and avoid making claims
        not supported by the data.""")

        return result,explaination.content
    
    except Exception as exp:
        return f"We got the Excpetion {exp}"


@tool
def get_graphs(query,code):
    """this is a function that which will give the  basic charts or visual summaries
    that which the user require based on the query and it will take code also
    how to plot based on tvariableshe and give only the df as dataframe not data """


    try:
        df= con.execute(f"""{query}""").fetchdf()
        explaination = llm.invoke(f"""
        SQL:{query}
        Query result:{df.to_string(index=False)}
        
        Explain the result shown in the chart.
        Mention important trends, highest/lowest values,
        percentages if relevant, and avoid making claims
        not supported by the data.""")
        exec(code)

        return Markdown(explaination.content)
    except Exception as exp:
        return f"We got the Excpetion {exp}"


In [21]:
from database import duckdb_connect


connect = duckdb_connect()

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", 
                             google_api_key=os.getenv("GOOGLE_API_KEY"),
                             temperature=0.7, 
                             max_tokens=None, timeout=None)



tools = [get_details_from_query,get_graphs]
llm_tool = llm.bind_tools(tools)


def get_response(question:str,db_describtion:list):
    messages = [
        SystemMessage(content=f"""You are an a high experienced duckdb sql assistnat where your task
        is to understand the text that which was provied by the user and you have all
        the metadate of the csv files that user require with their path and columns and its
        data type where your are an a sql assitant you are professional in ducksql that which
        will take csv file path instead of the table and remaing all will be same for example
        
        Example DuckDB Query: "SELECT * 
                               FROM read_csv_auto(s3://mybucket/categories.csv)
                               LIMIT 10" 
                               
        here you will pass the csv file and path read different read_csv_auto like this 
        you need to query that their might joints also where you need to pass two csv files instead 
        of tabel names  etc if you get any plot like design query in such a way that if it given to
        any ploting module it should execute  and result like that fit and create.  here is the users metadata
        {db_describtion}"""),

        HumanMessage(content=question)
        ]
    
    response = llm_tool.invoke(messages)

    if response.tool_calls:

        for tool_call in response.tool_calls:
            if tool_call['name'] == "get_details_from_query":
                return get_details_from_query.invoke({"query":(tool_call['args']['query'])})
            
            else:
                return get_graphs.invoke({"query": tool_call["args"]["query"],
                                          "code": tool_call["args"]["code"]})

    return response.content

In [16]:
question = "I want to ask questions about our CSV data and get useful business insights.”"

outs = get_response(question,file_describtion)